In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
BRANCH = "main"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("Current repo:", os.getcwd())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes peft transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train.py wad_dataset.py qformer_bridge.py scripts/test_infer.py scripts/prepare_qformer.py scripts/smoke_qformer_bridge.py


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config internvl_config.yaml


In [ ]:
!python scripts/smoke_qformer_bridge.py --config internvl_config.yaml


In [ ]:
!python train.py


In [ ]:
import subprocess
from pathlib import Path

CHECKPOINT_DIR = ""  # optional: fill manually with outputs/.../epoch_x or epoch_x_step_y
if not CHECKPOINT_DIR:
    candidates = sorted(Path("outputs/internvl3_2b").glob("*/*"), key=lambda p: p.stat().st_mtime)
    candidates = [p for p in candidates if (p / "adapter_config.json").exists()]
    CHECKPOINT_DIR = str(candidates[-1]) if candidates else ""

if CHECKPOINT_DIR:
    print("Evaluating checkpoint:", CHECKPOINT_DIR)
    subprocess.run([
        "python", "scripts/test_infer.py",
        "--config", "internvl_config.yaml",
        "--checkpoint", CHECKPOINT_DIR,
        "--split", "test_QA",
        "--output_file", "results/qformer_eval_test_QA.json",
    ], check=True)
else:
    print("No checkpoint found. Train first, then rerun this cell.")
